# reglscatterpy — interactive single-cell & spatial analysis

A walk through a real **single-cell pipeline** (10x PBMC 3k) and then a **412k-cell spatial** sample, using `reglscatterpy` as the interactive viewer at every step.

Plots are **interactive by default** (live WebGL widgets — lasso, zoom, filter, recolour). Pass `interactive=False` for a self-contained static snapshot, or `save_html(w, ...)` to export.

## Part 1 · Single-cell PBMCs

We start from the classic 10x **PBMC 3k** dataset, already through the standard scanpy pipeline
(QC → normalize → HVG → PCA → neighbours → UMAP → Louvain clustering). We'll explore it
interactively and annotate cell types.

In [ ]:
import scanpy as sc
import reglscatterpy as rs

adata = sc.read_h5ad("data/pbmc3k_processed.h5ad")
adata

### Quality control
Colour the embedding by QC metrics to spot low-quality cells (high mitochondrial fraction).

In [ ]:
rs.scatterplot(adata, basis="umap", color="percent_mito", cmap="magma")

Add distribution **sliders** to gate cells live (drag the black handles); `w.filtered` reads back
the original indices of the cells currently passing — over the whole dataset.

In [ ]:
w = rs.scatterplot(adata, basis="umap", color="louvain",
                   filter_by=["n_genes", "percent_mito"])
w

In [ ]:
# after dragging the sliders (or clicking legend categories):
len(w.filtered)        # cells passing the gates

### Clusters
The Louvain clusters scanpy found. `reglscatterpy` reuses scanpy's own colours
(`adata.uns['louvain_colors']`) in the same category order, so this matches `sc.pl.umap`.

In [ ]:
rs.scatterplot(adata, basis="umap", color="louvain")

Compare embeddings side by side — a **list of bases** makes a linked grid (pan/zoom/lasso stay in sync):

In [ ]:
rs.scatterplot(adata, basis=["umap", "tsne", "pca"], color="louvain")

### Marker genes
Colour by canonical markers to identify clusters — a **list of genes** gives one linked panel each.

In [ ]:
rs.scatterplot(adata, basis="umap",
               color=["MS4A1", "CST3", "NKG7", "PPBP"])   # B-cell, mono/DC, NK, platelet

A single marker with a scanpy-style **outline** (outer ring + inner gap) to make positive cells pop:

In [ ]:
rs.scatterplot(adata, basis="umap", color="CST3", cmap="viridis",
               add_outline=True, outline_color=("black", "white"), outline_width=(0.3, 0.05))

### Annotate cell types interactively
Lasso a cluster in the live plot, then inspect and label it. Here we set the selection from
Python (so the notebook is reproducible) — interactively you'd just lasso instead.

In [ ]:
w = rs.scatterplot(adata, basis="umap", color="louvain")
w

Select the B cells (lasso them, or programmatically), then see what's in the selection:

In [ ]:
w.selection = adata.obs_names[adata.obs["louvain"] == "B cells"]
w.selection.composition("louvain")        # counts + fractions in the selection

In [ ]:
w.diff_expression(n=5)                      # top markers of the selection vs rest (wilcoxon, scanpy)

In [ ]:
w.annotate("cell_type", "B cells")          # writes adata.obs['cell_type'] for those cells

Annotate the rest the same way, then **recolour by your annotation**. Cells you haven't labelled
yet show as light grey (a clear *NA* category), not the background:

In [ ]:
for lou, ct in {"CD4 T cells": "T cells", "CD8 T cells": "T cells",
                "CD14+ Monocytes": "Monocytes", "FCGR3A+ Monocytes": "Monocytes",
                "NK cells": "NK"}.items():
    w.selection = adata.obs_names[adata.obs["louvain"] == lou]
    w.annotate("cell_type", ct)

ct = rs.scatterplot(adata, basis="umap", color="cell_type")
ct

In [ ]:
# read the rendered palette and save it scanpy-style for reuse
ct.colors                                   # {category: '#rrggbb'}
adata.uns["cell_type_colors"] = list(ct.colors.values())

### Linked grid & export
`compose([...])` lays out plots in a synced grid; `save_html` writes a self-contained file.

In [ ]:
a = rs.scatterplot(adata, basis="umap", color="cell_type")
b = rs.scatterplot(adata, basis="umap", color="CST3")
rs.compose([a, b])

In [ ]:
import tempfile, os
out = os.path.join(tempfile.gettempdir(), "pbmc_umap.html")
rs.save_html(a, out)
out

## Part 2 · Spatial, at scale

A lung-cancer **spatial** sample: ~412k cells with both a UMAP and their tissue (`spatial`)
coordinates, plus cell-type and region annotations. (Point this path at your own file.)

In [ ]:
sp = sc.read_h5ad("/home/goguxor/Downloads/adata_sample.h5ad")
sp

### The tissue
Colour the spatial layout by cell type. `max_points=None` keeps **all** 412k cells resident.

In [ ]:
rs.scatterplot(sp, basis="spatial", color="cell_annotation_coarse", max_points=None)

### Big-data rendering
For very large atlases, `progressive=True` shows a density overview and streams full detail as you zoom in.

In [ ]:
rs.scatterplot(sp, basis="spatial", color="cell_annotation_coarse", progressive=True)

### UMAP ↔ tissue, linked
Show both layouts side by side. Lasso a region in the tissue and watch the same cells light up in the UMAP.

In [ ]:
rs.scatterplot(sp, basis=["umap", "spatial"], color="cell_annotation_coarse")

### ⭐ Morph between embeddings
Animate the **same cells** from the UMAP layout into their tissue positions — points *and* axes glide
together. Colour, selection, and filters carry through the transition (and stay recoverable).

In [ ]:
w = rs.scatterplot(sp, basis="umap", color="cell_annotation_coarse", max_points=None)
w

In [ ]:
w.morph_to("spatial", duration=2000)    # UMAP -> tissue

In [ ]:
w.morph_to("umap")                       # ...and back

Try it with state: lasso a cell type (or `w.selection = ...`), or drag a filter, **then** morph —
the highlight and filter persist across both layouts.

---
**Static vs interactive:** every plot above is a live widget by default (needs the Jupyter widgets
frontend). Pass `interactive=False` for a static HTML snapshot that renders anywhere, or
`rs.save_html(w, path)` to export. See the README for the full API.